# Multi-objective RM at 1.5B — the SCALE TEST (multi-head + ArmoRM gating)

At 0.5B, neither the fixed-weight multi-head (balanced 0.596) nor the ArmoRM prompt-only gating (0.589
overall) beat a curated single-scalar RM (v3 balanced 0.616 / flagship 1.5B 0.728). The honest hypothesis:
these methods need more scale + richer data. This runs the WHOLE multi-objective apparatus at **1.5B**:
3 specialist heads (helpful / safety / honesty) + prompt-only gating, then compares the fixed-weight
sweep AND the gated RM to the single-scalar flagship on RewardBench + RB2 Factuality. ~6-8 h on a T4.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE','->',('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1','torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -2   # includes multi-head + gating tests

## Config (3 objectives keyed by the data mix '@obj' tags)

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-1.5B-Instruct'
UF='argilla/ultrafeedback-binarized-preferences-cleaned'; PKU='PKU-Alignment/PKU-SafeRLHF'
# head0=helpful (UF+benign-scary), head1=safety (PKU), head2=honesty
MIX=f'{UF}:train[2000:6000]@0,data/benign_scary.jsonl:train@0,{PKU}:train[:4500]@1,data/honesty.jsonl:train@2'
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print('mix:',MIX)

## 1. Train the 3-head 1.5B RM (LoRA)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE -o model.use_lora=true \
  -o data.name="$MIX" -o data.train_split=train -o data.eval_split=none \
  -o data.max_length=512 -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 -o train.gradient_checkpointing=true \
  -o train.num_heads=3 -o train.bf16=$BF16 -o train.lr=1.0e-4 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o train.save_every=100000 -o output_dir=checkpoints/rm_mh3

## 2. Train the prompt-only gating (ArmoRM stage 2)

In [ ]:
!python scripts/train_gating.py --reward-model checkpoints/rm_mh3 --data "$MIX" \
  --output checkpoints/rm_mh3_gated --max-length 512 --batch-size 16 --device cuda --epochs 40 --lr 1e-3

## 3. Evaluate — fixed-weight sweep, gated RM, RB2 Factuality, probe

In [ ]:
import subprocess
def run(c):
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-1500:]);
    if o.returncode: print('ERR',o.stderr[-1000:])
    return o.stdout
sweep=run('python scripts/sweep_heads.py --reward-model checkpoints/rm_mh3 --device cuda --batch-size 32')
rb_g =run('python scripts/eval_rewardbench.py --reward-model checkpoints/rm_mh3_gated --device cuda --batch-size 32')
rb2_g=run('python scripts/eval_rewardbench2.py --reward-model checkpoints/rm_mh3_gated --subset Factuality --device cuda --batch-size 32')
prb  =run('python scripts/probe_honesty.py --reward-model checkpoints/rm_mh3_gated --device cuda')
md=('# Multi-objective RM @ 1.5B (scale test)\n\n'
    'Baselines: single-scalar 1.5B flagship RewardBench overall 0.728 (balanced safety 0.748); '
    '0.5B fixed-weight MH balanced 0.596; 0.5B gated 0.589.\n\n'
    '## Fixed-weight head sweep (RewardBench)\n```\n'+sweep[-1800:]+'\n```\n'
    '## Gated (ArmoRM prompt-only) RewardBench\n```\n'+rb_g[-900:]+'\n```\n'
    '## Gated RB2 Factuality\n```\n'+rb2_g[-500:]+'\n```\n'
    '## Gated honesty probe\n```\n'+prb[-500:]+'\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\nsaved RESULTS.md')

### Read
Does 1.5B lift the multi-objective methods above the single-scalar flagship (0.728 overall / balanced
0.748)? If the gated RM or a swept fixed-weight point clears it on the balanced axis, the scale
hypothesis holds and ArmoRM pays off; if not, the curated single-scalar remains best and the honest
conclusion is that these methods don't help at <=1.5B on this data.